In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"

silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
constructors_selected_df = constructors_df.select(
    F.col("constructorId").alias("constructor_id"),
    F.col("name").alias("constructor_name"),
    F.col("nationality"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [0]:
constructors_valid_df = constructors_selected_df.filter(
    F.col("constructor_id").isNotNull()
)

In [0]:
constructors_final_df = (
    constructors_valid_df
    .withColumn("nationality", F.initcap("nationality"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
    )

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
    constructors_final_df.write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(silver_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            constructors_final_df.alias("c"),
            "t.constructor_id = c.constructor_id"
        )
        .whenMatchedUpdate(
            condition="c.batch_id >= t.batch_id",
            set={
                "constructor_name": "c.constructor_name",
                "nationality": "c.nationality",
                "ingestion_timestamp": "c.ingestion_timestamp",
                "source_file": "c.source_file",
                "batch_id": "c.batch_id",
                "updated_at": "c.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )